# Member A — Segmentation, Robustness & Preferences

Reproduces the Member A analysis in the chapter: the three-cluster segmentation, its
robustness (silhouette, GMM/HDBSCAN agreement, bootstrap stability), and the content
preferences of active versus inactive users.

**Prerequisites** (run from repo root):
```bash
python -m src.data.build_labels
bash src/data/run_content_taste.sh "../Dataset/Raw_Data.zip"
python -m src.member_a_segmentation.segment
```
Deps: `pip3 install matplotlib scikit-learn jupyter`.


## Setup


In [15]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np, polars as pl
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, HDBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, adjusted_rand_score
from src import config
from src.member_a_segmentation.segment import load, matrix, FEATS

DER = config.DERIVED_DIR
plt.rcParams.update({"figure.figsize": (6.5, 4.2), "axes.grid": True, "grid.alpha": 0.3})
COLS = {1: "#B0B7BF", 2: "#2E5C8A", 0: "#C0392B"}
NAMES = {0: "Power users", 1: "Dormant majority", 2: "Engaged middle"}

df = load().join(pl.read_parquet(DER / "member_a" / "user_segments.parquet"), on="userId", how="inner")
Xs = matrix(df)
print(f"study users with segments: {df.height:,}")


study users with segments: 542,842


## 1. Segment profiles


In [16]:
prof = (df.group_by("segment").agg([
        pl.len().alias("n"),
        pl.col("is_inactive").mean().round(3).alias("inactive_rate"),
        pl.col("e_impr").mean().round(1).alias("early_impr"),
        pl.col("e_clicks").mean().round(2).alias("early_clicks"),
        pl.col("ct_click_n_content").mean().round(2).alias("content_cats"),
    ]).sort("segment"))
print(prof)


shape: (3, 6)
┌─────────┬────────┬───────────────┬────────────┬──────────────┬──────────────┐
│ segment ┆ n      ┆ inactive_rate ┆ early_impr ┆ early_clicks ┆ content_cats │
│ ---     ┆ ---    ┆ ---           ┆ ---        ┆ ---          ┆ ---          │
│ i32     ┆ u32    ┆ f64           ┆ f64        ┆ f64          ┆ f64          │
╞═════════╪════════╪═══════════════╪════════════╪══════════════╪══════════════╡
│ 0       ┆ 11035  ┆ 0.18          ┆ 250.3      ┆ 25.47        ┆ 9.28         │
│ 1       ┆ 459874 ┆ 0.772         ┆ 10.9       ┆ 0.05         ┆ 0.04         │
│ 2       ┆ 71933  ┆ 0.446         ┆ 42.1       ┆ 3.77         ┆ 2.44         │
└─────────┴────────┴───────────────┴────────────┴──────────────┴──────────────┘


## 2. PCA scatter of the segments


In [17]:
pca = PCA(n_components=2, random_state=42).fit(Xs)
Z = pca.transform(Xs); ev = pca.explained_variance_ratio_ * 100
seg = df["segment"].to_numpy()
rng = np.random.default_rng(42); idx = rng.choice(len(Z), 15000, replace=False)
plt.figure(figsize=(7, 5.2))
for s in [1, 2, 0]:
    m = seg[idx] == s
    plt.scatter(Z[idx][m, 0], Z[idx][m, 1], s=8, alpha=0.35 if s == 1 else 0.6, c=COLS[s], label=NAMES[s], edgecolors="none")
plt.xlabel(f"PC1 ({ev[0]:.0f}% variance)"); plt.ylabel(f"PC2 ({ev[1]:.0f}% variance)")
plt.title("User segments in the first two principal components")
plt.legend(markerscale=2); plt.tight_layout(); plt.show()


## 3. Robustness
Silhouette across k, agreement of GMM and HDBSCAN with k-means, and bootstrap stability.
(Uses a 50,000-user sample; the heavier methods take ~1 minute.)


In [18]:
S = Xs[np.random.default_rng(42).choice(len(Xs), 50000, replace=False)]
ks = range(2, 7); sil = []
for k in ks:
    km = KMeans(n_clusters=k, n_init=5, random_state=0).fit(S)
    sil.append(silhouette_score(S[:15000], km.labels_[:15000]))
plt.figure(); plt.plot(list(ks), sil, marker="o", color="#2E5C8A")
plt.xlabel("number of clusters k"); plt.ylabel("silhouette"); plt.title("Silhouette vs k"); plt.show()
for k, v in zip(ks, sil): print(f"  k={k}: silhouette={v:.3f}")


  k=2: silhouette=0.613
  k=3: silhouette=0.590
  k=4: silhouette=0.336
  k=5: silhouette=0.346
  k=6: silhouette=0.348


In [19]:
k3 = KMeans(n_clusters=3, n_init=10, random_state=42).fit(S).labels_
gmm = GaussianMixture(n_components=3, n_init=3, random_state=42).fit(S).predict(S)
hdb = HDBSCAN(min_cluster_size=2000, min_samples=50).fit_predict(S)
print(f"GMM(3) vs k-means(3): ARI = {adjusted_rand_score(k3, gmm):.3f}")
print(f"HDBSCAN: {len(set(hdb)) - (1 if -1 in hdb else 0)} clusters, "
      f"{(hdb == -1).mean():.1%} noise, ARI vs k-means = {adjusted_rand_score(k3, hdb):.3f}")

labs = [KMeans(n_clusters=3, n_init=3, random_state=b)
        .fit(S[np.random.default_rng(b).choice(len(S), len(S), replace=True)]).predict(S) for b in range(8)]
aris = [adjusted_rand_score(labs[i], labs[j]) for i in range(len(labs)) for j in range(i+1, len(labs))]
print(f"k=3 bootstrap stability: mean pairwise ARI = {np.mean(aris):.3f} (min {np.min(aris):.3f})")


GMM(3) vs k-means(3): ARI = 0.736
HDBSCAN: 12 clusters, 11.6% noise, ARI vs k-means = 0.031
k=3 bootstrap stability: mean pairwise ARI = 0.980 (min 0.958)


## 4. Do preferences distinguish users?
Part (a) asks what active users prefer. First, a caution: apparent taste *breadth* is confounded by volume.


In [20]:
ct = pl.read_parquet(DER / "user_content_taste.parquet").filter(pl.col("ct_clicked") > 0)
a = ct["ct_click_n_content"].to_numpy(); c = ct["ct_click_n_creators"].to_numpy(); n = ct["ct_clicked"].to_numpy()
print(f"corr(distinct content categories, #clicks) = {np.corrcoef(a, n)[0, 1]:.3f}")
print(f"corr(distinct creators, #clicks)           = {np.corrcoef(c, n)[0, 1]:.3f}")
print("=> 'broader taste' largely reduces to 'more clicks' - not a real preference signal.")


corr(distinct content categories, #clicks) = 0.845
corr(distinct creators, #clicks)           = 0.996
=> 'broader taste' largely reduces to 'more clicks' - not a real preference signal.


When volume is removed (each user described by her *composition* of clicks over the 122 content categories), no separable taste structure appears: a typical user puts only ~22% of clicks in her top category and spreads across ~20, and composition clusters are weak (silhouette < 0.18 vs 0.59 for the activity segmentation). True music genre was dropped from the dataset. See `src/member_a_segmentation/taste_analysis.py` for the full composition test.

**Conclusion:** users differ in *how much* they engage, not in cleanly separable *what* they prefer. Format (video vs image) also does not separate active from inactive users (~40% video clicks for both).


## Takeaways
- k=3 is highly stable (bootstrap ARI ~0.98) and matched by a Gaussian mixture (ARI ~0.74).
- HDBSCAN fragments the users: they form an engagement *gradient*, not discrete islands.
- The segmentation is **activity-based** (how much users engage), not taste-based.
- 'Broader taste' is a volume artifact (corr up to 0.996); volume-controlled composition shows no taste tribes.
- True genre is unavailable in the data; format does not distinguish active from inactive users.
